In [ ]:
import scvi
import muon as mu
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import re
import os

## Paths 

In [ ]:
OUTPUT_DIR = "results_terminal"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Read Data 

In [ ]:

mdata = md.read_h5mu("data/tonsil/tonsil_pp.hmu")
rna = mdata['rna']
protein = mdata['protein']

## Set keys

In [ ]:
SPVAE_LATENT_KEY = "X_spvae"

TOTALVI_CLUSTERS_KEY = "leiden_totalVI"
TOTALVI_CLUSTERS_KEY_LOUVAIN = "louvain_totalVI"
TOTALVI_CLUSTERS_KEY_KMEANS = "kmeans_totalVI"

## Running SpaVAE model 

In [ ]:
scvi.settings.seed = 0
scvi.model.SPVAE.setup_anndata(rna, batch_key="batch")   # or whatever batch key you have
model = scvi.model.SPVAE(rna)
model.train()

In [ ]:
rna.obsm[SPVAE_LATENT_KEY] = model.get_latent_representation()

In [ ]:
sc.pp.neighbors(rna, use_rep=SPVAE_LATENT_KEY)
sc.tl.umap(rna)

## Clustering 

In [ ]:
sc.tl.leiden(rna, key_added=TOTALVI_CLUSTERS_KEY, resolution=0.7)
sc.tl.louvain(rna, key_added=TOTALVI_CLUSTERS_KEY_LOUVAIN, resolution=0.7)

kmeans_mod = KMeans(n_clusters=5, random_state=0).fit(rna.obsm[SPVAE_LATENT_KEY])
rna.obs[TOTALVI_CLUSTERS_KEY_KMEANS] = kmeans_mod.labels_.astype(str)

## UMAP Plot 

In [ ]:
rna_umap_plot = mu.pl.embedding(
    mdata,
    basis="RNA:X_umap",
    color=[f"RNA:{TOTALVI_CLUSTERS_KEY_KMEANS}",
           f"RNA:{TOTALVI_CLUSTERS_KEY_LOUVAIN}",
           f"RNA:{TOTALVI_CLUSTERS_KEY}"],
    frameon=False,
    palette="Accent",
    title=["K-means", "Louvain", "Leiden"],
    return_fig=True
)
rna_umap_plot.savefig(os.path.join(OUTPUT_DIR, "spVAE_cluster_UMAP.pdf"))
rna_umap_plot.savefig(os.path.join(OUTPUT_DIR, "spVAE_cluster_UMAP.png", dpi=300))

## Moran's I (spatial autocorrelation on embeddings)

In [ ]:
moran_i = sc.metrics.morans_i(rna, obsm=SPVAE_LATENT_KEY)
pd.DataFrame({"moran_i": moran_i}).to_csv(os.path.join(OUTPUT_DIR, "moran_i.csv"))

## LISI

In [ ]:
def compute_lisi(emb, labels, perplexity=30):
    n_neighbors = int(3 * perplexity)
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(emb)
    distances, indices = nn.kneighbors(emb)
    indices = indices[:, 1:]

    labels = np.array(labels)
    out = []
    for i in range(emb.shape[0]):
        neigh = labels[indices[i]]
        p = pd.value_counts(neigh) / len(neigh)
        entropy = -(p * np.log(p)).sum()
        out.append(np.exp(entropy))
    return np.array(out)

rna.obs["cLISI_kmeans5"] = compute_lisi(rna.obsm[SPVAE_LATENT_KEY], rna.obs[TOTALVI_CLUSTERS_KEY_KMEANS])
rna.obs["cLISI_louvain"] = compute_lisi(rna.obsm[SPVAE_LATENT_KEY], rna.obs[TOTALVI_CLUSTERS_KEY_LOUVAIN])
rna.obs["cLISI_leiden"] = compute_lisi(rna.obsm[SPVAE_LATENT_KEY], rna.obs[TOTALVI_CLUSTERS_KEY])

rna.obs[['cLISI_kmeans5','cLISI_louvain','cLISI_leiden']].to_csv(os.path.join(OUTPUT_DIR, "LISI_metrics.csv"))

## Differential expression 

In [ ]:
de_df = model.differential_expression(
    groupby=TOTALVI_CLUSTERS_KEY,
    delta=0.5,
    batch_correction=True
)

de_df.to_csv(os.path.join(OUTPUT_DIR, "differential_expression.csv"))

## Filter RNA + Protein Markers 

In [ ]:
filtered_pro = {}
filtered_rna = {}

cats = rna.obs[TOTALVI_CLUSTERS_KEY].cat.categories

for c in cats:
    cid = f"{c} vs Rest"
    cell_type_df = de_df.loc[de_df.comparison == cid]
    cell_type_df = cell_type_df.sort_values("lfc_median", ascending=False)
    cell_type_df = cell_type_df[cell_type_df.lfc_median > 0]

    pro_rows = cell_type_df.index.str.contains("protein")
    data_pro = cell_type_df.iloc[pro_rows]
    data_pro = data_pro[data_pro["bayes_factor"] > 0.7]

    data_rna = cell_type_df.iloc[~pro_rows]
    data_rna = data_rna[data_rna["bayes_factor"] > 3]
    data_rna = data_rna[data_rna["non_zeros_proportion1"] > 0.1]

    filtered_pro[c] = data_pro.index.tolist()[:3]
    filtered_rna[c] = data_rna.index.tolist()[:2]

## Heatmap 

In [ ]:
sc.tl.dendrogram(rna, groupby=TOTALVI_CLUSTERS_KEY, use_rep=SPVAE_LATENT_KEY)

gene_list = [x for sub in filtered_rna.values() for x in sub]

sc.pl.heatmap(
    rna,
    var_names=gene_list,
    groupby=TOTALVI_CLUSTERS_KEY,
    use_raw=False,
    log=True
)

plt.savefig(os.path.join(OUTPUT_DIR, "RNA_heatmap.png"), dpi=300)

In [ ]:
protein.obs[TOTALVI_CLUSTERS_KEY] = rna.obs[TOTALVI_CLUSTERS_KEY]
protein.obsm[SPVAE_LATENT_KEY] = rna.obsm[SPVAE_LATENT_KEY]

sc.tl.dendrogram(protein, groupby=TOTALVI_CLUSTERS_KEY, use_rep=SPVAE_LATENT_KEY)

filtered_pro_cleaned = [x for sub in filtered_pro.values() for x in sub]
filtered_pro_cleaned = [re.sub(r"\.[ATGC]*_protein", "", g) for g in filtered_pro_cleaned]
sc.pl.heatmap(
        protein,
        var_names=filtered_pro_cleaned,
        groupby=TOTALVI_CLUSTERS_KEY,
        use_raw=False,
        log=True,
        show=False
    )
plt.savefig(os.path.join(OUTPUT_DIR, "protein_heatmap.png"), dpi=300)


## Normalization + LOG FOR RANK GENES 

In [ ]:
sc.pp.normalize_total(rna)
sc.pp.log1p(rna)

## Wilcoxon Rank 

In [ ]:
sc.tl.rank_genes_groups(rna, TOTALVI_CLUSTERS_KEY, method="wilcoxon", key_added="wilcoxon")
sc.pl.rank_genes_groups(rna, n_genes=25, sharey=False, key="wilcoxon")

## Dotplot 

In [ ]:
rna_dotplot = sc.pl.rank_genes_groups_dotplot(
    rna,
    n_genes=8,
    values_to_plot="logfoldchanges",
    key="wilcoxon",
    groupby=TOTALVI_CLUSTERS_KEY,
    cmap="RdBu_r",
    vmin=-1.25,
    vcenter=0,
    vmax=1.25,
    return_fig=True
)

rna_dotplot.fig.set_size_inches(18, 6)
rna_dotplot.fig.savefig(os.path.join(OUTPUT_DIR, "RNA_dotplot.pdf"), bbox_inches="tight")
rna_dotplot.fig.savefig(os.path.join(OUTPUT_DIR, "RNA_dotplot.png"), dpi=300, bbox_inches="tight")

## Save updated Mu Data 

In [ ]:
mdata['rna'] = rna
mdata['protein'] = protein

mdata.write_h5mu(os.path.join(OUTPUT_DIR, "tonsil_spvae_results.h5mu"))

print(f"All results saved in {OUTPUT_DIR}")

## ARI and NMI Scores

In [ ]:
#Rand Index (ARI) is a function that computes the similarity between two data clusterings by comparing predicted labels with true labels.
#ARI = 0 indicates random labeling, while ARI =1 indicates perfect agreement between the clusterings
#negative ARI values indicate less agreement than expected by chance, lowest possible = -0.5 

#Normalized Mutual Information (NMI) is a measure of similarity between two clustering results
# Normalized to scale from 0 (no mutual information) to 1(perfect correlation)

from sklearn import metrics 

labels_true = output_x
labels_pred = output_y

RI = metrics.rand_score(labels_true, labels_pred)
ARI = metrics.adjusted_rand_score(labels_true, labels_pred)
NMI = metrics.normalized_mutual_info_score(labels_true, labels_pred)

print(RI, ARI, NMI)

